# Parameters: settings and records

`ariel.parameters` is where ARIEL keeps its fixed numbers and the base classes of its records. There are three kinds of things in it: **settings** (`ArielModulesConfig`: masses, sizes and servo limits of the robot modules, `MujocoConfig`, `ArielConfig`) that can be overridden from environment variables; **record bases** (`FrozenModel`, `MutableModel`) that every pydantic record in ARIEL derives from; and **type aliases** (`Position`, `FloatArray`, ...).

In this notebook you read the module settings, override one with an environment variable, see how the two record bases treat changes and unknown fields, and check the array types. No graphics backend is needed.

In [1]:
# Standard library
import os

In [2]:
# Third-party libraries
import numpy as np
import pydantic
from rich.table import Table

In [3]:
# Local libraries
from ariel import console
from ariel.parameters import (
    ND_FLOAT_PRECISION,
    ND_INT_PRECISION,
    ArielConfig,
    ArielModulesConfig,
    FloatArray,
    FrozenModel,
    IntArray,
    MujocoConfig,
    MutableModel,
)

Nothing here is random, so there is no seed.

## The module settings

`ArielModulesConfig` holds what every brick and hinge is made of: the mass and size of a brick, and the specification of the servo that drives a hinge (`ACTUATOR`).

In [4]:
modules = ArielModulesConfig()
table = Table(title="ArielModulesConfig")
table.add_column("setting")
table.add_column("value")
table.add_row("BRICK_MASS (kg)", str(modules.BRICK_MASS))
table.add_row("BRICK_DIMENSIONS (m)", str(modules.BRICK_DIMENSIONS))
for name, value in modules.ACTUATOR:
    table.add_row(f"ACTUATOR.{name}", str(value))
console.print(table)

                ArielModulesConfig                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ setting                   ┃ value              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ BRICK_MASS (kg)           │ 0.055              │
│ BRICK_DIMENSIONS (m)      │ (0.05, 0.05, 0.05) │
│ ACTUATOR.MIN_ANGLE        │ -3.141592653589793 │
│ ACTUATOR.MAX_ANGLE        │ 3.141592653589793  │
│ ACTUATOR.MAX_SPEED        │ 0.18               │
│ ACTUATOR.STALL_TORQUE     │ 13.5               │
│ ACTUATOR.MAX_TORQUE_POWER │ 15.0               │
│ ACTUATOR.DIMENSIONS       │ (54.5, 20.0, 47.5) │
│ ACTUATOR.MIN_FREQUENCY    │ 50.0               │
│ ACTUATOR.MAX_FREQUENCY    │ 330.0              │
└───────────────────────────┴────────────────────┘

In [5]:
assert modules.BRICK_MASS > 0
assert all(side > 0 for side in modules.BRICK_DIMENSIONS)
assert modules.ACTUATOR.MIN_ANGLE < 0 < modules.ACTUATOR.MAX_ANGLE
assert modules.ACTUATOR.MIN_FREQUENCY < modules.ACTUATOR.MAX_FREQUENCY
assert MujocoConfig().floor_name == "floor"

## Override a setting from the environment

These classes are `pydantic-settings` classes, so a setting is read from the environment variable of the same name when one is set. The next cell sets `BRICK_MASS` and `VERBOSE`, builds the configs, and removes the variables again (in `finally`, so a failure cannot leave them behind).

In [6]:
os.environ["BRICK_MASS"] = "0.1"
os.environ["VERBOSE"] = "false"
try:
    heavy = ArielModulesConfig()
    quiet = ArielConfig()
finally:
    del os.environ["BRICK_MASS"]
    del os.environ["VERBOSE"]
console.print(
    f"brick mass {modules.BRICK_MASS} -> {heavy.BRICK_MASS} kg; "
    f"verbose {ArielConfig().verbose} -> {quiet.verbose}"
)

brick mass 0.055 -> 0.1 kg; verbose True -> False

In [7]:
assert heavy.BRICK_MASS > modules.BRICK_MASS
assert not quiet.verbose
assert ArielConfig().verbose  # the variable is gone again

## The two record bases

A `FrozenModel` cannot change after it is created. A `MutableModel` can. Both reject a field they do not know, which catches a misspelt name instead of silently ignoring it.

In [8]:
class Frozen(FrozenModel):
    size: int


class Mutable(MutableModel):
    size: int


NEW_SIZE = 2
frozen, mutable = Frozen(size=1), Mutable(size=1)
mutable.size = NEW_SIZE
try:
    frozen.size = NEW_SIZE
except pydantic.ValidationError as error:
    console.print(f"frozen: {error.errors()[0]['type']}")
try:
    Mutable(size=1, sise=2)
except pydantic.ValidationError as error:
    console.print(f"unknown field: {error.errors()[0]['type']}")

frozen: frozen_instance

unknown field: extra_forbidden

In [9]:
assert mutable.size == NEW_SIZE
assert frozen.size == 1
rejected = []
for model in (Frozen, Mutable):
    try:
        model(size=1, sise=2)
    except pydantic.ValidationError:
        rejected.append(model.__name__)
assert rejected == ["Frozen", "Mutable"]

## Array types

`FloatArray` and `IntArray` are the aliases ARIEL uses in type hints, with fixed precision: 64-bit floats and 32-bit integers. The constants say which dtype to ask numpy for.

In [10]:
positions: FloatArray = np.zeros((4, 3), dtype=ND_FLOAT_PRECISION)
indices: IntArray = np.arange(4, dtype=ND_INT_PRECISION)
console.print(f"positions {positions.dtype}, indices {indices.dtype}")
assert positions.dtype == np.float64
assert indices.dtype == np.int32

positions float64, indices int32